> **Not executed by the repo author: run in Colab or in your course lab; versions may need updates.**
> Package pins below were chosen from memory of releases that existed; if a pin fails, install without the version and note what you got.

# Lab 9: Backdoor attacks with BackdoorBox

**Goal:** poison an image training set with a small patch trigger (BadNets), train a classifier, and measure clean accuracy versus attack success.

- **Runtime:** Colab with a GPU (T4). BackdoorBox is a research toolbox; it may need a GPU and specific library versions.
- **Time:** 20 to 40 minutes.
- **Offline twin:** `../labs/09-backdoor-attacks/` poisons a text dataset, detects the trigger and cleans up, on a laptop.
- **Hedge:** BackdoorBox is maintained on GitHub (search for "BackdoorBox THU-DBL"). Its install steps and API can change. Read the repository README and its `tests/` examples first, and treat the code below as an outline, not a guaranteed recipe.
- **Safety note:** the trigger here is a harmless white square on a public dataset. Do not poison data you do not own or share poisoned models.

## Step 1: Get the toolbox
Clone the repository and install its requirements as its README describes (do not trust a pin from this notebook).

In [ ]:
!git clone https://github.com/THU-DBL/BackdoorBox.git
%cd BackdoorBox
!pip install -q -r requirements.txt

## Step 2: Datasets
CIFAR-10 is small and standard. The toolbox expects torchvision datasets with a loader.

In [ ]:
import os, torch, torch.nn as nn
from torchvision import transforms
from torchvision.datasets import CIFAR10, DatasetFolder
import core  # BackdoorBox package at the repo root

tf = transforms.Compose([transforms.ToTensor()])
train_ds = CIFAR10(root="data", train=True, transform=tf, download=True)
test_ds = CIFAR10(root="data", train=False, transform=tf, download=True)

## Step 3: Define a BadNets attack
A 3x3 white patch in the bottom-right corner is the trigger; poisoned images get label `y_target`. Argument names below follow the toolbox examples as we remember them; check the repo's `tests/test_BadNets.py` for the current signature.

In [ ]:
pattern = torch.zeros((32, 32), dtype=torch.uint8); pattern[-3:, -3:] = 255
weight = torch.zeros((32, 32), dtype=torch.float32); weight[-3:, -3:] = 1.0

badnets = core.BadNets(
    train_dataset=train_ds, test_dataset=test_ds,
    model=core.models.ResNet(18), loss=nn.CrossEntropyLoss(),
    y_target=1, poisoned_rate=0.05, pattern=pattern, weight=weight,
    seed=666, deterministic=True)

## Step 4: Train and test
The schedule below is a short one. Real runs use more epochs.

In [ ]:
schedule = {
    "device": "GPU", "CUDA_VISIBLE_DEVICES": "0", "GPU_num": 1,
    "benign_training": False, "batch_size": 128, "num_workers": 2,
    "lr": 0.1, "momentum": 0.9, "weight_decay": 5e-4, "gamma": 0.1, "schedule": [15, 20],
    "epochs": 25, "log_iteration_interval": 100, "test_epoch_interval": 5,
    "save_epoch_interval": 25, "save_dir": "experiments", "experiment_name": "badnets_cifar10"}
badnets.train(schedule)

## Step 5: Read the two numbers
- **Clean accuracy** (benign test set): should stay close to a normal model.
- **Attack success rate** (test images with the trigger classified as `y_target`): high for a working backdoor.

Compare with the offline lab: same pattern, different data type.

## Step 6: Defense thinking
The toolbox also ships defenses (see its README for the current list). Try one, then answer: what data checks, provenance rules and model scans would have caught this before deployment? See `02-llm-security/model-threats/` for the repo's checklist.